# Disease Prediction - Week 10
Works with **any CSV / Excel dataset** up to **1000 rows** and **100 columns**.

### Step 1: Import Libraries

In [ ]:
!pip install -q xgboost

import pandas as pd
import numpy as np
import zipfile
import warnings
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

warnings.filterwarnings('ignore')

# ---------- Settings ----------
MAX_ROWS = 1000     # maximum patients (rows) allowed
MAX_COLS = 100      # maximum attributes (columns) allowed
TARGET_COL = None   # name of the result column. Keep None to detect it automatically

### Step 2: Upload Dataset
Upload a `.csv`, `.xlsx` or `.zip` (with a CSV inside).

In [ ]:
from google.colab import files
uploaded = files.upload()

fname = list(uploaded.keys())[0]
if fname.endswith('.zip'):
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('.')
        fname = [f for f in z.namelist() if f.endswith('.csv')][0]

if fname.endswith(('.xlsx', '.xls')):
    df = pd.read_excel(fname)
else:
    df = pd.read_csv(fname)

df.columns = df.columns.astype(str).str.strip()

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

print("Total rows loaded:", df.shape[0])
print("Total columns loaded:", df.shape[1])

if df.shape[0] > MAX_ROWS or df.shape[1] > MAX_COLS:
    raise ValueError(f"Dataset too large. Allowed: {MAX_ROWS} rows and {MAX_COLS} columns. "
                     f"Your file has {df.shape[0]} rows and {df.shape[1]} columns.")
df

### Step 3: Find the Result Column (Target)
The program looks for a column named *classification, class, target, label, diagnosis, outcome, result* or *status*. If none is found, the **last column** is used.

To choose it yourself, set `TARGET_COL = 'your_column_name'` in Step 1.

In [ ]:
if TARGET_COL is None:
    lower_names = {c.lower(): c for c in df.columns}
    for name in ['classification', 'class', 'target', 'label', 'diagnosis', 'outcome', 'result', 'status']:
        if name in lower_names:
            TARGET_COL = lower_names[name]
            break
    else:
        TARGET_COL = df.columns[-1]

if TARGET_COL not in df.columns:
    raise ValueError(f"Column '{TARGET_COL}' not found. Available columns: {list(df.columns)}")

print("Result column (target):", TARGET_COL)

### Step 4: Clean Data

In [ ]:
# 1. Remove ID-type columns
id_names = ['id', 'patient id', 'patient_id', 'patient no', 'sr no', 's.no', 'sno', 'index']
drop_cols = [c for c in df.columns if c != TARGET_COL and
             (c.lower() in id_names or c.lower().startswith('unnamed'))]

# 2. Remove text columns where every value is different (like names or IDs)
for c in df.columns:
    if c != TARGET_COL and not pd.api.types.is_numeric_dtype(df[c]) and df[c].nunique() == len(df):
        drop_cols.append(c)

df = df.drop(columns=list(set(drop_cols)))
print("Removed ID columns:", list(set(drop_cols)))

# 3. Clean text values (spaces, tabs, '?', 'nan' etc.)
missing_marks = ['?', '', 'nan', 'NaN', 'NAN', 'None', 'NA', 'N/A', 'null']
for col in df.columns:
    if not pd.api.types.is_numeric_dtype(df[col]):
        df[col] = df[col].astype(str).str.strip().replace(missing_marks, np.nan)

# 4. Remove columns that are fully empty, and rows with no result
df = df.dropna(axis=1, how='all')
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)

# 5. Detect number columns and text columns automatically
feature_cols = [c for c in df.columns if c != TARGET_COL]
num_cols, cat_cols = [], []

for col in feature_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        num_cols.append(col)
    else:
        converted = pd.to_numeric(df[col], errors='coerce')
        if converted.notna().sum() >= 0.9 * df[col].notna().sum():
            df[col] = converted
            num_cols.append(col)
        else:
            cat_cols.append(col)

# 6. Fill missing values
for col in num_cols:
    df[col] = df[col].astype(float).fillna(df[col].astype(float).median())
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

print("Number columns:", len(num_cols))
print("Category columns:", len(cat_cols))
print("Missing values left:", int(df.isnull().sum().sum()))

### Step 5: Encode Text Columns

In [ ]:
for col in cat_cols:
    df[col] = LabelEncoder().fit_transform(df[col].astype(str))

# Encode the result column and remember the names
target_encoder = LabelEncoder()
y = pd.Series(target_encoder.fit_transform(df[TARGET_COL].astype(str)), name=TARGET_COL)
X = df.drop(columns=[TARGET_COL])

print("Result classes:")
for i, name in enumerate(target_encoder.classes_):
    print(f"  {i} = {name}")
print()
print(y.value_counts())

### Step 6: Train-Test Split

In [ ]:
# Stratify only when every class has at least 2 patients
use_stratify = y if y.value_counts().min() >= 2 else None

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=use_stratify)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])

### Train-Test Split Explained + Save Data

The dataset is divided into:

- **80% → Training Data** — Teaches the model
- **20% → Testing Data** — Checks model performance

Both parts are saved as separate CSV files and downloaded to your laptop below.

In [ ]:
train_data = X_train.copy()
train_data[TARGET_COL] = y_train.values

test_data = X_test.copy()
test_data[TARGET_COL] = y_test.values

train_data.to_csv("train_data.csv", index=False)
test_data.to_csv("test_data.csv", index=False)

print("Training data saved:", train_data.shape)
print("Testing data saved:", test_data.shape)

from google.colab import files as colab_files
colab_files.download("train_data.csv")
colab_files.download("test_data.csv")

### Step 7: Train 4 Models and Calculate All 5 Parameters

Each model is checked with the same five evaluation parameters used in the report:
**Accuracy, Precision, Recall, F1-Score, and AUC-ROC** — all computed from the 20% test data.

In [ ]:
models = {
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(random_state=42),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(max_iter=1000)
}

n_classes = y.nunique()
average_type = 'binary' if n_classes == 2 else 'weighted'

results = {}
predictions = {}

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        pred = model.predict(X_test_scaled)
        proba = model.predict_proba(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        pred = model.predict(X_test)
        proba = model.predict_proba(X_test)

    acc = accuracy_score(y_test, pred)
    prec = precision_score(y_test, pred, average=average_type, zero_division=0)
    rec = recall_score(y_test, pred, average=average_type, zero_division=0)
    f1 = f1_score(y_test, pred, average=average_type, zero_division=0)

    # AUC-ROC: binary uses the positive-class probability,
    # multi-class (like Dataset 5's 5 stages) uses a weighted one-vs-rest score
    try:
        if n_classes == 2:
            auc = roc_auc_score(y_test, proba[:, 1])
        else:
            auc = roc_auc_score(y_test, proba, multi_class='ovr', average='weighted')
    except ValueError:
        auc = np.nan

    results[name] = {
        "Accuracy (%)": round(acc * 100, 2),
        "Precision (%)": round(prec * 100, 2),
        "Recall (%)": round(rec * 100, 2),
        "F1-Score (%)": round(f1 * 100, 2),
        "AUC-ROC (%)": round(auc * 100, 2) if not np.isnan(auc) else np.nan,
    }
    predictions[name] = pred

### Step 8: Results — All 5 Parameters

In [ ]:
results_df = pd.DataFrame(results).T.reset_index().rename(columns={"index": "Model"})
results_df = results_df.sort_values(by="Accuracy (%)", ascending=False).reset_index(drop=True)
results_df

### Step 9: Disease Detection Result for All Patients

In [ ]:
best_model_name = results_df.iloc[0]["Model"]
best_model = models[best_model_name]

if best_model_name == "Logistic Regression":
    X_full_input = scaler.transform(X)
else:
    X_full_input = X

full_pred = best_model.predict(X_full_input)

output = pd.DataFrame({
    "Actual": target_encoder.inverse_transform(y.values),
    "Predicted": target_encoder.inverse_transform(full_pred),
    "Data Used": np.where(X.index.isin(X_test.index), "Test", "Train")
})

print("Best Model:", best_model_name)
print("  Accuracy :", results[best_model_name]["Accuracy (%)"], "%")
print("  Precision:", results[best_model_name]["Precision (%)"], "%")
print("  Recall   :", results[best_model_name]["Recall (%)"], "%")
print("  F1-Score :", results[best_model_name]["F1-Score (%)"], "%")
print("  AUC-ROC  :", results[best_model_name]["AUC-ROC (%)"], "%")
print("Total patients shown:", len(output))
output

### Save & Download Final Result

In [ ]:
output.to_csv("final_result.csv", index_label="Patient No")

print("Final result saved:", output.shape)

from google.colab import files as colab_files
colab_files.download("final_result.csv")